# AI-NIDS: Anomaly Detection Validation
## Isolation Forest Anomaly Identification & Ground Truth Verification
This notebook trains an Isolation Forest model to detect network intrusions and validates whether the identified anomalies are **actual anomalies (attacks)** or **false alarms (benign traffic)** using dataset ground truth labels.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import IsolationForest
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    precision_recall_curve,
    f1_score,
    precision_score,
    recall_score,
    roc_curve
)

# Set plot styles
sns.set_theme(style="whitegrid")
plt.rcParams['font.size'] = 11

In [ ]:
# 1. Load Dataset
data_path = 'D:/dulith_doc/peojects/AI_NIDS/MachineLearningCVE/Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv'
df1 = pd.read_csv(data_path)

# Clean column names (strip whitespace)
df1.columns = df1.columns.str.strip()
target_col = 'Label'

print(f"Target Column: '{target_col}'")
print("Class Distribution in Raw Data:")
print(df1[target_col].value_counts())

# Extract ground truth labels (Binary: 1 = Attack / Anomaly, 0 = BENIGN / Normal)
y_binary = (df1[target_col] != 'BENIGN').astype(int)
y_raw = df1[target_col]

# 2. Extract numeric features
numeric_df = df1.select_dtypes(include=[np.number])

# Drop zero-variance (constant) columns
constant_cols = [col for col in numeric_df.columns if numeric_df[col].nunique() <= 1]
df_cleaned = numeric_df.drop(columns=constant_cols)
print(f"\nDropped {len(constant_cols)} constant columns.")

# Drop redundant (highly correlated > 0.95) columns
corr_matrix = df_cleaned.corr().abs()
upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
redundant_cols = [col for col in upper_tri.columns if any(upper_tri[col] > 0.95)]
df_cleaned = df_cleaned.drop(columns=redundant_cols)
print(f"Dropped {len(redundant_cols)} redundant columns.")

# Handle missing / infinite values while maintaining strict index alignment
df_cleaned = df_cleaned.replace([np.inf, -np.inf], np.nan)
valid_idx = df_cleaned.dropna().index

df_cleaned = df_cleaned.loc[valid_idx]
y_binary = y_binary.loc[valid_idx]
y_raw = y_raw.loc[valid_idx]

print("\n" + "*"*10 + " Cleaned Dataset Summary " + "*"*10)
print(f"Cleaned Feature Matrix Shape: {df_cleaned.shape}")
print(f"Total BENIGN (Normal) Samples: {(y_binary == 0).sum():,}")
print(f"Total Attack (Anomaly) Samples: {(y_binary == 1).sum():,}")

In [ ]:
# 1. Stratified Train/Val/Test Split (60% Train, 20% Validation, 20% Test)
X_train_val, X_test, y_train_val, y_test, y_raw_train_val, y_raw_test = train_test_split(
    df_cleaned, y_binary, y_raw, test_size=0.20, random_state=42, stratify=y_binary
)

X_train, X_val, y_train, y_val, y_raw_train, y_raw_val = train_test_split(
    X_train_val, y_train_val, y_raw_train_val, test_size=0.25, random_state=42, stratify=y_train_val
)

# 2. Fit RobustScaler ONLY on X_train to prevent data leakage
scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train)

# 3. Transform validation and test sets using fitted scaler
X_val_scaled  = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print(f"Total Dataset Shape:       {df_cleaned.shape}")
print(f"Training Set Shape (60%):  {X_train.shape}  (Attacks: {(y_train == 1).sum():,})")
print(f"Validation Set Shape (20%): {X_val.shape}  (Attacks: {(y_val == 1).sum():,})")
print(f"Test Set Shape (20%):       {X_test.shape}  (Attacks: {(y_test == 1).sum():,})")

In [ ]:
# Initialize Isolation Forest
model = IsolationForest(
    n_estimators=300,       # 300 trees to evaluate features thoroughly
    contamination='auto',   # Use 'auto' during fit; tune threshold on validation set
    random_state=42,
    n_jobs=-1
)

# FIT ONLY ON TRAINING SET
print("Fitting Isolation Forest model on X_train...")
model.fit(X_train_scaled)
print("Model training complete.")

# Get continuous anomaly scores for validation data (lower/negative = more anomalous)
val_scores = model.decision_function(X_val_scaled)

# Determine threshold based on validation set anomaly ratio
anomaly_ratio_val = (y_val == 1).mean() * 100
threshold_percentile = np.percentile(val_scores, anomaly_ratio_val)

# Compute F1-optimal threshold on validation set
precisions, recalls, thresholds = precision_recall_curve(y_val, -val_scores)
f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-8)
best_f1_idx = np.argmax(f1_scores)
threshold_f1 = -thresholds[best_f1_idx] if best_f1_idx < len(thresholds) else threshold_percentile

# Set decision threshold
threshold = threshold_f1
print(f"Validation Set Anomaly Ratio: {anomaly_ratio_val:.2f}%")
print(f"Selected Anomaly Score Threshold: {threshold:.4f} (Best Val F1-Score: {f1_scores[best_f1_idx]:.4f})")

In [ ]:
# 1. Get continuous anomaly decision scores on unseen test data
test_scores = model.decision_function(X_test_scaled)

# 2. Flag samples with decision scores below threshold as predicted anomalies (1 = Anomaly, 0 = Normal)
test_pred_anomalies = (test_scores < threshold).astype(int)

# 3. Ground Truth Verification & Evaluation Metrics
total_test = len(y_test)
num_flagged = np.sum(test_pred_anomalies)

cm = confusion_matrix(y_test, test_pred_anomalies)
tn, fp, fn, tp = cm.ravel()

precision = precision_score(y_test, test_pred_anomalies)
recall = recall_score(y_test, test_pred_anomalies)
f1 = f1_score(y_test, test_pred_anomalies)
roc_auc = roc_auc_score(y_test, -test_scores)

print("="*65)
print("       IDENTIFIED ANOMALIES GROUND TRUTH VERIFICATION REPORT")
print("="*65)
print(f"Total Test Samples Evaluated:            {total_test:,}")
print(f"Total Anomalies Flagged by Model:         {num_flagged:,} ({num_flagged/total_test:.2%})")
print("-"*65)
print(f"True Positives  (Actual Attacks Flagged): {tp:,}  ({tp/num_flagged:.2%} of flagged anomalies)")
print(f"False Positives (BENIGN False Alarms):    {fp:,}  ({fp/num_flagged:.2%} of flagged anomalies)")
print(f"True Negatives  (BENIGN Correctly Passed):{tn:,}")
print(f"False Negatives (Attacks Missed):        {fn:,}")
print("-"*65)
print(f"Precision (Accuracy of Flagged Anomalies): {precision:.4f} ({precision*100:.2f}% of flagged anomalies are real attacks)")
print(f"Recall (Attack Detection Rate):           {recall:.4f} ({recall*100:.2f}% of all real attacks detected)")
print(f"F1-Score:                                  {f1:.4f}")
print(f"ROC-AUC Score:                             {roc_auc:.4f}")
print("="*65)

print("\nClassification Report:")
print(classification_report(y_test, test_pred_anomalies, target_names=['BENIGN (Normal)', 'ATTACK (Anomaly)']))

# Detailed Breakdown of Flagged Anomalies by Raw Label
df_flagged = pd.DataFrame({
    'Actual_Label': y_raw_test[test_pred_anomalies == 1]
})
print("\nBreakdown of Flagged Anomalies by Actual Traffic Type:")
print(df_flagged['Actual_Label'].value_counts())

In [ ]:
# 1. Setup Diagnostic Visualizations
plt.figure(figsize=(16, 12))

# Subplot 1: Confusion Matrix Heatmap
plt.subplot(2, 2, 1)
sns.heatmap(cm, annot=True, fmt=',d', cmap='Blues', cbar=False,
            xticklabels=['Predicted Normal', 'Predicted Anomaly'],
            yticklabels=['Actual BENIGN', 'Actual Attack'])
plt.title('Confusion Matrix: Identified vs Actual Anomalies', fontsize=13, fontweight='bold')
plt.xlabel('Model Prediction')
plt.ylabel('Ground Truth')

# Subplot 2: Verification of Flagged Anomalies (True vs False Anomalies)
plt.subplot(2, 2, 2)
flagged_counts = pd.Series({'True Anomalies (Attacks)': tp, 'False Alarms (BENIGN)': fp})
colors = ['#2ca02c', '#d62728']
bars = plt.bar(flagged_counts.index, flagged_counts.values, color=colors, width=0.45)
plt.title(f'Verification of {num_flagged:,} Flagged Anomalies', fontsize=13, fontweight='bold')
plt.ylabel('Number of Samples')
for bar in bars:
    height = bar.get_height()
    pct = height / num_flagged * 100 if num_flagged > 0 else 0
    plt.text(bar.get_x() + bar.get_width()/2., height + (max(tp, fp)*0.02),
             f'{height:,}\n({pct:.1f}%)', ha='center', va='bottom', fontweight='bold')
plt.ylim(0, max(tp, fp) * 1.25)

# Subplot 3: Anomaly Score Distribution (BENIGN vs Attack Traffic)
plt.subplot(2, 2, 3)
sns.kdeplot(test_scores[y_test == 0], label='Actual BENIGN (Normal)', color='blue', fill=True, alpha=0.3)
sns.kdeplot(test_scores[y_test == 1], label='Actual Attack (Anomaly)', color='red', fill=True, alpha=0.3)
plt.axvline(x=threshold, color='black', linestyle='--', linewidth=2, label=f'Threshold ({threshold:.4f})')
plt.title('Isolation Forest Score Distribution', fontsize=13, fontweight='bold')
plt.xlabel('Anomaly Score (Lower = More Anomalous)')
plt.ylabel('Density')
plt.legend()

# Subplot 4: Precision-Recall Curve
plt.subplot(2, 2, 4)
prec_curve, rec_curve, _ = precision_recall_curve(y_test, -test_scores)
plt.plot(rec_curve, prec_curve, color='purple', linewidth=2, label=f'Precision-Recall (F1={f1:.4f})')
plt.title('Precision-Recall Curve', fontsize=13, fontweight='bold')
plt.xlabel('Recall (Detection Rate)')
plt.ylabel('Precision (True Anomaly Ratio)')
plt.legend()
plt.grid(True, linestyle=':', alpha=0.6)

plt.tight_layout()
plt.show()